
# Vibe Matcher — Prototype Notebook (Deliverable)

**One-paragraph intro (why AI at Nexora?):**

Nexora aims to deliver personalized, delightful shopping experiences by connecting users' ephemeral "vibes" — mood-driven, short textual queries like "energetic urban chic" — with products that match that feeling. An AI-driven Vibe Matcher uses semantic embeddings to represent product descriptions and user queries in a shared vector space, enabling fast, relevance-aware retrieval. This prototype demonstrates the end-to-end approach (data prep → embeddings → vector search → evaluation) and provides an easy path to production improvements (Pinecone/Weaviate, model upgrades, user feedback loop), helping Nexora elevate conversion and discovery.

---

This notebook contains:
- Mock product data
- Embedding stubs (OpenAI comment + TF-IDF fallback)
- Cosine-similarity ranking (top-3)
- Evaluation over 3 queries and a simple latency plot
- Notes on improvements & deployment ideas

**Submission deadline:** 11 November 2025


In [ ]:

# Imports & data prep
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time
import matplotlib.pyplot as plt
from timeit import default_timer as timer

# Mock product catalog (7 items)
products = [
    {"id": 1, "name": "Boho Dress", "desc": "Flowy midi dress, earthy tones, tassels — perfect for festival and bohemian vibes.", "tags":["boho","festival","flowy"]},
    {"id": 2, "name": "Urban Bomber Jacket", "desc": "Cropped bomber with reflective trim, structured shoulders — energetic urban chic.", "tags":["urban","chic","edgy"]},
    {"id": 3, "name": "Cozy Knit Sweater", "desc": "Oversized knit sweater, neutral palette, fuzzy texture — cozy and warm for relaxed days.", "tags":["cozy","casual","warm"]},
    {"id": 4, "name": "Sleek Slip Dress", "desc": "Silky slip dress with minimalist cut — night-out elegant and understated.", "tags":["elegant","night-out","minimalist"]},
    {"id": 5, "name": "Street Runner Sneakers", "desc": "Lightweight sneakers with bold accents and high-grip sole — active, streetwear energy.", "tags":["sporty","street","active"]},
    {"id": 6, "name": "Vintage Tote Bag", "desc": "Canvas tote with retro patches and warm tones — vintage, artistic, casual.", "tags":["vintage","artsy","casual"]},
    {"id": 7, "name": "Tailored Blazer", "desc": "Structured blazer in slate grey — professional yet modern, perfect for sharp looks.", "tags":["professional","modern","sharp"]}
]

df = pd.DataFrame(products)
df['text_for_embedding'] = df['name'] + '. ' + df['desc'] + ' Tags: ' + df['tags'].apply(lambda x: ','.join(x))
df


In [ ]:

# Embedding helpers (OpenAI stub + TF-IDF fallback)
def get_openai_embeddings(texts, model='text-embedding-ada-002', api_key=None):
    # Stub: Example code to run in Colab/local with internet:
    # import openai
    # openai.api_key = api_key
    # resp = openai.Embedding.create(model=model, input=texts)
    # vectors = [r['embedding'] for r in resp['data']]
    # return np.array(vectors)
    raise RuntimeError("OpenAI embedding call is disabled in this environment. Use TF-IDF fallback or run in Colab with network access.")

def get_tfidf_embeddings(texts, max_features=512):
    vect = TfidfVectorizer(max_features=max_features, ngram_range=(1,2))
    X = vect.fit_transform(texts)
    return X.toarray(), vect

# Build TF-IDF vectors for product texts
prod_texts = df['text_for_embedding'].tolist()
prod_vectors, tfidf_vect = get_tfidf_embeddings(prod_texts)
print('Product vectors shape (TF-IDF fallback):', prod_vectors.shape)


In [ ]:

# Similarity search: top-k via cosine similarity
def match_query_tfidf(query, prod_vectors, prod_df, k=3, vect=None):
    if vect is None:
        raise ValueError('Provide a fitted TF-IDF vectorizer for query transformation in offline mode.')
    q_vec = vect.transform([query]).toarray()
    sims = cosine_similarity(q_vec, prod_vectors)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    results = []
    for i, idx in enumerate(top_idx):
        results.append({
            'rank': i+1,
            'id': int(prod_df.iloc[idx]['id']),
            'name': prod_df.iloc[idx]['name'],
            'desc': prod_df.iloc[idx]['desc'],
            'score': float(sims[idx])
        })
    return results, sims

# Demo match for the sample query
sample_query = 'energetic urban chic'
results, sims = match_query_tfidf(sample_query, prod_vectors, df, k=3, vect=tfidf_vect)
print('Query:', sample_query)
for r in results:
    print(r['rank'], r['name'], '(score={:.4f})'.format(r['score']))


In [ ]:

# Edge handling: fallback when top score below threshold
def match_with_fallback(query, prod_vectors, prod_df, vect, k=3, threshold=0.35):
    results, sims = match_query_tfidf(query, prod_vectors, prod_df, k=k, vect=vect)
    top_score = results[0]['score']
    if top_score < threshold:
        fallback_prompt = ("No close match found. Consider expanding the query or use this assisted prompt:\n"
                           "Find products that match '{}' — try adding context like 'casual/party/formal' or color cues.".format(query))
        return {'results': results, 'top_score': top_score, 'fallback': fallback_prompt}
    return {'results': results, 'top_score': top_score, 'fallback': None}

edge_query = 'glitter runway couture fantasy'
edge_out = match_with_fallback(edge_query, prod_vectors, df, tfidf_vect, k=3, threshold=0.35)
print('Edge query top score:', edge_out['top_score'])
if edge_out['fallback']:
    print('Fallback suggestion:', edge_out['fallback'])
else:
    for r in edge_out['results']:
        print(r['rank'], r['name'], '(score={:.4f})'.format(r['score']))


In [ ]:

# Run 3 queries and evaluate (with sim > 0.7 considered 'good' for this proxy)
queries = ['energetic urban chic', 'cozy weekend at home', 'minimal night-out elegance']

log = []
latencies = []

for q in queries:
    start = timer()
    out = match_with_fallback(q, prod_vectors, df, tfidf_vect, k=3, threshold=0.35)
    end = timer()
    latency = end - start
    latencies.append(latency)
    top_score = out['top_score']
    is_good = top_score > 0.7
    log.append({'query': q, 'top_score': top_score, 'is_good': is_good, 'latency_s': latency})
    print('\nQuery:', q)
    print('Latency (s): {:.4f}'.format(latency))
    if out['fallback']:
        print('Fallback:', out['fallback'])
    for r in out['results']:
        print(r['rank'], r['name'], '(score={:.4f})'.format(r['score']))

eval_df = pd.DataFrame(log)
print('\nEvaluation summary:')
print(eval_df)

# Plot latency and save
plt.figure(figsize=(6,3))
plt.plot(range(1,len(latencies)+1), latencies, marker='o')
plt.xlabel('Query #')
plt.ylabel('Latency (s)')
plt.title('Vibe Matcher - Query Latency (TF-IDF proxy)')
plt.grid(True)
plt.tight_layout()
plot_path = '/mnt/data/vibe_matcher_latency.png'
plt.savefig(plot_path)
plt.close()

# Save evaluation CSV
eval_csv_path = '/mnt/data/vibe_matcher_eval.csv'
eval_df.to_csv(eval_csv_path, index=False)

print('\nSaved evaluation CSV to:', eval_csv_path)
print('Saved latency plot to:', plot_path)


In [ ]:

# Reflection & improvements (3-5 bullets)
reflection = [
    "Replace TF-IDF fallback with OpenAI embeddings (text-embedding-ada-002) for true semantic matching.",
    "Use a vector DB (Pinecone/Weaviate/Chroma) for scalable nearest-neighbor search and persistent indices.",
    "Collect user feedback (clicked / purchased) to fine-tune retrieval and re-rank with a learning-to-rank model.",
    "Add hybrid retrieval: tag-based filtering + vector similarity to respect inventory constraints (size/color).",
    "Handle cold-start and multilingual queries by adding query expansion and language detection steps."
]
for b in reflection:
    print('-', b)


In [ ]:

# Instructions to run with OpenAI embeddings (Colab / local environment with internet)
instructions = (
"1. pip install openai\n"
"2. export OPENAI_API_KEY=\"your_api_key\"  # or set via environment in Colab\n"
"3. Uncomment and adapt the get_openai_embeddings() snippet in the notebook.\n"
"4. Replace TF-IDF embedding creation:\n       prod_vectors = get_openai_embeddings(prod_texts, api_key=OPENAI_API_KEY)\n"
"   and transform the query similarly:\n       q_vec = get_openai_embeddings([query], api_key=OPENAI_API_KEY)\n"
"5. Normalize vectors (optional) and compute cosine similarity as shown.\n"
"6. For production, push product vectors to Pinecone / Weaviate and use approximate NN for low latency and scaling.\n"
)
print(instructions)
